# Da planilha à trilha ótima — experimentos sobre os dados reais do GuIA

Este notebook refaz os experimentos do trabalho **a partir das planilhas de dados
brutos** (`report/dados-brutos-guia.xlsx`), ponto a ponto:

1. ler as tabelas por curso (uma aba por curso, módulo a módulo);
2. transformar cada módulo em um **item da mochila 0/1** (peso = tempo medido, valor = score);
3. definir os orçamentos de tempo do aluno (capacidade T);
4. rodar os algoritmos do repo (`algorithms/`) e comparar: quem acha a trilha ótima, a que custo;
5. montar a tabela **módulos × algoritmos** — o embrião das colunas que vão entrar na planilha.

Tudo aqui usa o mesmo código dos experimentos oficiais (`experiment_real.py`);
a diferença é a narrativa célula a célula.

In [1]:
import pandas as pd

from algorithms import (
    LearningModule, evaluate,
    exhaustive_dfs, dp_table, dp_vector, dp_memoized,
    greedy, greedy_with_best_item, branch_and_bound,
    meet_in_the_middle, fptas, grasp,
)

PLANILHA = "report/dados-brutos-guia.xlsx"
ABAS = ["Curso 2", "Curso 3", "Curso 4"]

## 1. Ler as tabelas da planilha

Cada aba tem um título na linha 1 e o cabeçalho na linha 3 (`header=2`).
As colunas são as que montamos juntas: tempo previsto × medido, nº de tópicos,
os três scores e os contadores de alunos.

In [2]:
tabelas = {}
for aba in ABAS:
    df = pd.read_excel(PLANILHA, sheet_name=aba, header=2)
    df = df[df["módulo"].astype(str).str.startswith("C")]  # descarta a linha de total
    tabelas[aba] = df
tabelas["Curso 2"]

,módulo,ordem,tempo previsto (min),tempo medido (min),nº tópicos,score conclusão (%),score conteúdo,score NPS (x10),iniciaram,concluíram
0,C2M1,1.0,não declarado,109.7,6.0,34,6.0,95.0,368.0,126.0
1,C2M2,2.0,não declarado,204.5,12.0,28,12.0,95.0,101.0,28.0
2,C2M3,3.0,não declarado,376,13.0,26,13.0,95.0,72.0,19.0
3,C2M4,4.0,não declarado,69.6,6.0,62,6.0,95.0,48.0,30.0
4,C2M5,5.0,não declarado,104.2,8.0,68,8.0,95.0,28.0,19.0
5,C2M6,6.0,não declarado,sem medição,1.0,—,1.0,95.0,17.0,0.0


In [3]:
tabelas["Curso 3"]

,módulo,ordem,tempo previsto (min),tempo medido (min),nº tópicos,score conclusão (%),score conteúdo,score NPS (x10),iniciaram,concluíram
0,C3M1,1.0,120.0,34.6,6.0,43,6.0,92.0,261.0,113.0
1,C3M2,2.0,120.0,46.1,5.0,44,5.0,92.0,139.0,61.0
2,C3M3,3.0,120.0,16.5,3.0,49,3.0,92.0,103.0,50.0
3,C3M4,4.0,120.0,sem medição,1.0,—,1.0,92.0,44.0,0.0


In [4]:
tabelas["Curso 4"]

,módulo,ordem,tempo previsto (min),tempo medido (min),nº tópicos,score conclusão (%),score conteúdo,score NPS (x10),iniciaram,concluíram
0,C4M1,1.0,não declarado,28.7,6.0,62,6.0,98.0,71.0,44.0
1,C4M2,2.0,não declarado,32,5.0,71,5.0,98.0,45.0,32.0
2,C4M3,3.0,não declarado,21.7,4.0,56,4.0,98.0,62.0,35.0
3,C4M4,4.0,não declarado,63.6,5.0,76,5.0,98.0,25.0,19.0
4,C4M5,5.0,não declarado,sem medição,1.0,—,1.0,98.0,27.0,0.0


## 2. Da linha da tabela ao item da mochila

Um item precisa de **peso** e **valor** inteiros:

- **peso** = `tempo medido (min)`, arredondado pro minuto inteiro (mínimo 1).
  Linha `sem medição` **fica de fora** — sem peso válido não há item (é por isso
  que a instância tem n = 12 e não 15).
- **valor** = um dos três scores da planilha. O catálogo inteiro vira uma única
  instância (trilha pode cruzar cursos), como nos experimentos oficiais.

In [5]:
def itens_da_planilha(score_col):
    """Catálogo inteiro -> lista de LearningModule (peso=min medidos, valor=score)."""
    itens = []
    for aba, df in tabelas.items():
        for _, linha in df.iterrows():
            tempo = pd.to_numeric(linha["tempo medido (min)"], errors="coerce")
            valor = pd.to_numeric(linha[score_col], errors="coerce")
            if pd.isna(tempo) or pd.isna(valor):
                continue  # 'sem medição' / '—'
            itens.append(LearningModule(linha["módulo"], max(1, round(float(tempo))), int(valor)))
    return itens

SCORES = {
    "conclusao": "score conclusão (%)",
    "conteudo": "score conteúdo",
    "nps": "score NPS (x10)",
}

itens = {nome: itens_da_planilha(col) for nome, col in SCORES.items()}
pd.DataFrame([(m.name, m.time, m.score) for m in itens["conteudo"]],
             columns=["módulo", "peso (min)", "valor (score conteúdo)"])

,módulo,peso (min),valor (score conteúdo)
0,C2M1,110,6
1,C2M2,204,12
2,C2M3,376,13
3,C2M4,70,6
4,C2M5,104,8
5,C3M1,35,6
6,C3M2,46,5
7,C3M3,16,3
8,C4M1,29,6
9,C4M2,32,5


Confirmação rápida: 12 itens, pesos de 17 a 376 min (o *spread* de 23×), e a soma
dos pesos — o catálogo inteiro custa ~18,5 h, contra orçamentos de 2 a 10 h/semana.
**É o problema do paper em uma linha: não cabe tudo, é preciso escolher.**

In [6]:
pesos = [m.time for m in itens["conteudo"]]
print(f"n = {len(pesos)} itens | peso mín = {min(pesos)} min | máx = {max(pesos)} min "
      f"({max(pesos)/min(pesos):.0f}x) | soma = {sum(pesos)} min = {sum(pesos)/60:.1f} h")

n = 12 itens | peso mín = 16 min | máx = 376 min (24x) | soma = 1108 min = 18.5 h


## 3. Os orçamentos do aluno (capacidade T)

Os mesmos dos experimentos: 2 h, 5 h, 10 h e 20 h por semana, em minutos.

In [7]:
ORCAMENTOS = {"2h": 120, "5h": 300, "10h": 600, "20h": 1200}
ORCAMENTOS

{'2h': 120, '5h': 300, '10h': 600, '20h': 1200}

## 4. Rodar os algoritmos

Todos recebem a MESMA instância e devolvem um `Result(value, indices, operations)`.
O contador de operações é o custo (passos elementares, independente de hardware).

Primeiro o retrato completo num cenário: **score conteúdo, aluno com 2 h** —
a versão "vinda da planilha" da Tabela IV do paper.

In [8]:
ALGORITMOS = [
    ("DFS exaustivo", lambda M, T: exhaustive_dfs(M, T)),
    ("DP (tabela)", lambda M, T: dp_table(M, T)[0]),
    ("DP (vetor)", lambda M, T: dp_vector(M, T)),
    ("DP (memoizada)", lambda M, T: dp_memoized(M, T)),
    ("B&B", lambda M, T: branch_and_bound(M, T)),
    ("MITM", lambda M, T: meet_in_the_middle(M, T)),
    ("FPTAS (eps=0,1)", lambda M, T: fptas(M, T, 0.1)),
    ("Greedy", lambda M, T: greedy(M, T)),
    ("Greedy + melhor item", lambda M, T: greedy_with_best_item(M, T)),
    ("GRASP", lambda M, T: grasp(M, T, seed=0)),
]

def rodada(modulos, T):
    linhas = []
    otimo = dp_table(modulos, T)[0].value
    for nome, f in ALGORITMOS:
        r = f(modulos, T)
        reconstroi = bool(r.indices) or r.value == 0
        tempo_usado, _ = evaluate(modulos, r.indices)
        linhas.append({
            "algoritmo": nome,
            "score da trilha": r.value,
            "ótimo": otimo,
            "perdeu": f"{100 * (otimo - r.value) / otimo:.1f}%" if otimo else "0.0%",
            "tempo usado (min)": tempo_usado if reconstroi else "—",
            "operações": r.operations,
            "módulos escolhidos": " ".join(modulos[i].name for i in sorted(r.indices))
                                  if reconstroi else "— (só o valor; não reconstrói)",
        })
    return pd.DataFrame(linhas)

rodada(itens["conteudo"], ORCAMENTOS["2h"])

,algoritmo,score da trilha,ótimo,perdeu,tempo usado (min),operações,módulos escolhidos
0,DFS exaustivo,21,21,0.0%,118,245,C3M1 C4M1 C4M2 C4M3
1,DP (tabela),21,21,0.0%,118,1452,C3M1 C4M1 C4M2 C4M3
2,DP (vetor),21,21,0.0%,—,682,— (só o valor; não reconstrói)
3,DP (memoizada),21,21,0.0%,118,154,C3M1 C4M1 C4M2 C4M3
4,B&B,21,21,0.0%,118,113,C3M1 C4M1 C4M2 C4M3
5,MITM,21,21,0.0%,118,218,C3M1 C4M1 C4M2 C4M3
6,"FPTAS (eps=0,1)",21,21,0.0%,118,6740,C3M1 C4M1 C4M2 C4M3
7,Greedy,19,21,9.5%,102,40,C3M1 C3M3 C4M1 C4M3
8,Greedy + melhor item,19,21,9.5%,102,52,C3M1 C3M3 C4M1 C4M3
9,GRASP,21,21,0.0%,118,2823,C3M1 C4M1 C4M2 C4M3


Leitura: **todos os exatos acham a mesma trilha ótima**; o greedy para antes.
B&B é o exato mais barato; a DP paga mais operações porque o custo dela depende
do valor numérico de T (é o "NP-difícil em sentido fraco" do paper); o DFS do
RUTICO cresce 2^n e só é viável porque n = 12.

## 5. A tabela "módulos × algoritmos"

A visão que vai pra planilha: cada módulo numa linha, cada algoritmo numa coluna,
✓ = entrou na trilha daquele algoritmo. Rodapé: tempo usado, score e custo.
(O DP vetor fica de fora desta visão: por projeto ele só devolve o valor ótimo,
sem reconstruir a trilha — economiza memória, perde a lista.)

In [9]:
def modulos_x_algoritmos(modulos, T):
    tab = pd.DataFrame(index=[m.name for m in modulos])
    tab.insert(0, "peso (min)", [m.time for m in modulos])
    tab.insert(1, "valor", [m.score for m in modulos])
    rodape = {}
    for nome, f in ALGORITMOS:
        if nome == "DP (vetor)":
            continue  # devolve só o valor ótimo, não a trilha
        r = f(modulos, T)
        escolhidos = set(r.indices)
        tab[nome] = ["✓" if i in escolhidos else "·" for i in range(len(modulos))]
        t_usado, _ = evaluate(modulos, r.indices)
        rodape[nome] = (t_usado, r.value, r.operations)
    nomes = [nome for nome, _ in ALGORITMOS if nome != "DP (vetor)"]
    for rotulo, pos in [("tempo usado (min)", 0), ("score obtido", 1), ("operações", 2)]:
        tab.loc[rotulo] = ["", ""] + [rodape[nome][pos] for nome in nomes]
    return tab

modulos_x_algoritmos(itens["conteudo"], ORCAMENTOS["2h"])

,peso (min),valor,DFS exaustivo,DP (tabela),DP (memoizada),B&B,MITM,"FPTAS (eps=0,1)",Greedy,Greedy + melhor item,GRASP
C2M1,110,6,·,·,·,·,·,·,·,·,·
C2M2,204,12,·,·,·,·,·,·,·,·,·
C2M3,376,13,·,·,·,·,·,·,·,·,·
C2M4,70,6,·,·,·,·,·,·,·,·,·
C2M5,104,8,·,·,·,·,·,·,·,·,·
C3M1,35,6,✓,✓,✓,✓,✓,✓,✓,✓,✓
C3M2,46,5,·,·,·,·,·,·,·,·,·
C3M3,16,3,·,·,·,·,·,·,✓,✓,·
C4M1,29,6,✓,✓,✓,✓,✓,✓,✓,✓,✓
C4M2,32,5,✓,✓,✓,✓,✓,✓,·,·,✓


## 6. Varredura completa: 3 scores × 4 orçamentos

DP (exato, referência) contra greedy (rápido e arriscado) em todos os cenários —
o "quanto se ganha por otimizar de verdade".

In [10]:
linhas = []
for score, modulos in itens.items():
    for rotulo, T in ORCAMENTOS.items():
        dp = dp_table(modulos, T)[0]
        gr = greedy(modulos, T)
        t_dp, _ = evaluate(modulos, dp.indices)
        t_gr, _ = evaluate(modulos, gr.indices)
        gap = 100 * (dp.value - gr.value) / dp.value if dp.value else 0.0
        linhas.append({
            "score": score, "orçamento": rotulo,
            "ótimo (DP)": dp.value, "greedy": gr.value, "gap greedy": f"{gap:.1f}%",
            "tempo DP (min)": t_dp, "tempo greedy (min)": t_gr,
            "ops DP": dp.operations, "ops greedy": gr.operations,
        })
pd.DataFrame(linhas)

,score,orçamento,ótimo (DP),greedy,gap greedy,tempo DP (min),tempo greedy (min),ops DP,ops greedy
0,conclusao,2h,238,238,0.0%,99,99,1452,38
1,conclusao,5h,420,401,4.5%,279,244,3612,38
2,conclusao,10h,565,565,0.0%,528,528,7212,38
3,conclusao,20h,619,619,0.0%,1108,1108,14412,38
4,conteudo,2h,21,19,9.5%,118,102,1452,40
5,conteudo,5h,37,35,5.4%,284,250,3612,40
6,conteudo,10h,56,54,3.6%,600,528,7212,40
7,conteudo,20h,79,79,0.0%,1108,1108,14412,40
8,nps,2h,386,386,0.0%,118,99,1452,38
9,nps,5h,671,668,0.4%,298,244,3612,38


## 7. Uma rodada da DP responde todos os orçamentos

Propriedade da Seção VII-A: a tabela da DP calculada para o maior T já contém a
resposta ótima de QUALQUER orçamento menor — útil pro produto (o aluno muda o
tempo disponível e a trilha se refaz sem custo extra).

In [11]:
modulos = itens["conclusao"]
curva = []
for T in range(60, 1201, 60):
    r = dp_table(modulos, T)[0]
    curva.append({"T (min)": T, "T (h)": T / 60, "score ótimo": r.value,
                  "módulos": " ".join(modulos[i].name for i in sorted(r.indices))})
pd.DataFrame(curva)

,T (min),T (h),score ótimo,módulos
0,60,1.0,127,C4M2 C4M3
1,120,2.0,238,C3M3 C4M1 C4M2 C4M3
2,180,3.0,325,C3M1 C3M2 C3M3 C4M1 C4M2 C4M3
3,240,4.0,376,C2M4 C3M3 C4M1 C4M2 C4M3 C4M4
4,300,5.0,420,C2M4 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4
5,360,6.0,469,C2M5 C3M1 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4
6,420,7.0,531,C2M4 C2M5 C3M1 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4
7,480,8.0,531,C2M4 C2M5 C3M1 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4
8,540,9.0,565,C2M1 C2M4 C2M5 C3M1 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4
9,600,10.0,565,C2M1 C2M4 C2M5 C3M1 C3M2 C3M3 C4M1 C4M2 C4M3 C4M4


---
**Onde isso se liga ao resto do repo:** o protocolo completo (validação sintética,
instância de tópicos n = 78, GRASP, contagens pareadas) está em `experiment_real.py`
e `experiment_validation.py`; os resultados oficiais em `results/real_*.csv`. Este
notebook é a trilha didática: planilha → instância → algoritmos → tabela final.